# DICE — one-run RSL-RL workflow

This notebook installs DICE in an isolated Python 3.12 environment and launches the complete final task directly. There is no curriculum or staged training path.

> **Run this notebook from a fresh Colab runtime.** The earlier notebook mixed a CUDA 13 PyTorch installation with Isaac Lab's CUDA 12.8 stack. In Colab, use **Runtime → Disconnect and delete runtime**, reconnect to an NVIDIA GPU runtime, and then run these cells from top to bottom.


## 1. Mount Drive and obtain the repository

This cell only clones or updates the repository. The DICE package is installed after Isaac Sim, Isaac Lab, PyTorch, and RSL-RL are made consistent.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
from pathlib import Path

REPO = Path('/content/dice')

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/djdhillxn/dice.git', str(REPO)],
        check=True,
    )

print(f'DICE repository: {REPO}')


## 2. Create an isolated Isaac environment

The notebook invokes this environment's Python explicitly for installation, training, evaluation, and rendering. It does not modify or depend on Colab's preinstalled PyTorch stack.

Set `REBUILD_ENV = True` only when you intentionally want to delete and rebuild the environment.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

VENV = Path('/content/dice_isaac_env')
PYTHON = VENV / 'bin' / 'python'
REBUILD_ENV = False

if REBUILD_ENV and VENV.exists():
    shutil.rmtree(VENV)

if not PYTHON.exists():
    subprocess.run(['python3.12', '-m', 'venv', str(VENV)], check=True)

os.environ['OMNI_KIT_ACCEPT_EULA'] = 'YES'
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'


def run(command, cwd=None, env=None):
    command = [str(part) for part in command]
    print('\n$', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True)


print(f'Isolated Python: {PYTHON}')


## 3. Install the compatible Isaac Sim and PyTorch stack

Isaac Sim 6.0.1 uses Python 3.12. On Linux x86-64, the matching Isaac Lab setup uses PyTorch 2.11 with CUDA 12.8. PyTorch is installed **after** Isaac Sim so any generic CUDA build selected by pip is replaced by the CUDA-12.8 wheel.


In [ ]:
run([
    PYTHON, '-m', 'pip', 'install', '--upgrade',
    'pip', 'setuptools<82', 'wheel',
])

run([
    PYTHON, '-m', 'pip', 'install', '--no-cache-dir',
    'isaacsim[all,extscache]==6.0.1.0',
    '--extra-index-url', 'https://pypi.nvidia.com',
])

# Remove any generic PyPI PyTorch build selected by Isaac Sim before installing
# the CUDA-12.8 build required by this Linux x86-64 Isaac Lab workflow.
run([
    PYTHON, '-m', 'pip', 'uninstall', '-y',
    'torch', 'torchvision', 'torchaudio',
])

run([
    PYTHON, '-m', 'pip', 'install', '--no-cache-dir',
    'torch==2.11.0', 'torchvision==0.26.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128',
])


## 4. Install Isaac Lab with only the required RSL-RL and Kit components

The old `./isaaclab.sh --install` command attempted to install optional imitation-learning packages and failed while building `egl_probe`. DICE does not need those packages. This cell follows the selective installer path and lets Isaac Lab choose its compatible RSL-RL dependency.

Do **not** add a separate `pip install rsl-rl-lib` cell afterward; that was what replaced the CUDA-12.8 PyTorch build with CUDA 13 in the failed run.


In [ ]:
ISAACLAB = Path('/content/IsaacLab')

if ISAACLAB.exists():
    run(['git', '-C', ISAACLAB, 'fetch', 'origin', 'develop', '--depth', '1'])
    run(['git', '-C', ISAACLAB, 'checkout', '-B', 'develop', 'origin/develop'])
    run(['git', '-C', ISAACLAB, 'reset', '--hard', 'origin/develop'])
else:
    run([
        'git', 'clone', '--depth', '1', '--branch', 'develop',
        'https://github.com/isaac-sim/IsaacLab.git', ISAACLAB,
    ])

install_env = os.environ.copy()
install_env['VIRTUAL_ENV'] = str(VENV)
install_env['PATH'] = f"{VENV / 'bin'}:{install_env.get('PATH', '')}"
install_env['OMNI_KIT_ACCEPT_EULA'] = 'YES'

run(
    [ISAACLAB / 'isaaclab.sh', '-i', 'rl [rsl-rl],visualizer [kit]'],
    cwd=ISAACLAB,
    env=install_env,
)


## 5. Install DICE


In [ ]:
run([PYTHON, '-m', 'pip', 'install', '-e', f'{REPO}[video]'])


## 6. Verify the CUDA runtime used by training

This is a single installation check for the exact failure seen previously. It confirms that PyTorch is using CUDA 12.8, that the matching NVRTC built-ins library exists, and that a CUDA normalization operation can execute before Isaac Sim is launched.


In [ ]:
site_packages = subprocess.check_output(
    [str(PYTHON), '-c', 'import site; print(site.getsitepackages()[0])'],
    text=True,
).strip()

nvidia_lib_dirs = sorted(
    str(path)
    for path in (Path(site_packages) / 'nvidia').glob('*/lib')
    if path.is_dir()
)

RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['VIRTUAL_ENV'] = str(VENV)
RUNTIME_ENV['PATH'] = f"{VENV / 'bin'}:{RUNTIME_ENV.get('PATH', '')}"
RUNTIME_ENV['OMNI_KIT_ACCEPT_EULA'] = 'YES'
RUNTIME_ENV['LD_LIBRARY_PATH'] = ':'.join(
    nvidia_lib_dirs + [RUNTIME_ENV.get('LD_LIBRARY_PATH', '')]
).rstrip(':')

verification = """
import importlib.metadata as metadata
from pathlib import Path
import site
import torch

print('torch:', torch.__version__)
print('torch CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('RSL-RL:', metadata.version('rsl-rl-lib'))
print('Isaac Lab:', metadata.version('isaaclab'))

if torch.version.cuda != '12.8':
    raise RuntimeError(
        f'Expected the CUDA 12.8 PyTorch build, found {torch.version.cuda}. '
        'Rebuild /content/dice_isaac_env from a fresh Colab runtime.'
    )

root = Path(site.getsitepackages()[0]) / 'nvidia' / 'cuda_nvrtc' / 'lib'
builtins = sorted(root.glob('libnvrtc-builtins.so.12*'))
if not builtins:
    raise RuntimeError(f'CUDA 12 NVRTC built-ins were not found under {root}.')
print('NVRTC built-ins:', builtins[-1])

x = torch.tensor([[1.0, 2.0, 3.0]], device='cuda')
y = torch.nn.functional.normalize(x, dim=-1)
torch.cuda.synchronize()
print('CUDA normalization:', y.cpu().tolist())
print('CUDA installation check passed.')
"""

run([PYTHON, '-c', verification], env=RUNTIME_ENV)


## 7. The Full Monty run

The default AppLauncher behavior is headless when no visualizer is requested, so the deprecated `--headless` argument is intentionally omitted.


In [ ]:
NUM_ENVS = 2048
MAX_ITERATIONS = 10_000
RUN_NAME = 'final'

run([
    PYTHON,
    REPO / 'scripts' / 'train_rsl.py',
    '--task', 'DICE-Shadow-Train-v0',
    '--num_envs', str(NUM_ENVS),
    '--max_iterations', str(MAX_ITERATIONS),
    '--run_name', RUN_NAME,
], cwd=REPO, env=RUNTIME_ENV)


## 8. Select the checkpoint

Use the final checkpoint or a regular checkpoint selected through nominal evaluation. Replace `<run>` with the generated timestamped directory name.


In [ ]:
CHECKPOINT = REPO / 'outputs' / 'DICE' / '<run>' / 'model_final.pt'
print(CHECKPOINT)


## 9. Nominal and robust evaluation


In [ ]:
run([
    'bash', REPO / 'scripts' / 'run_final_evaluation.sh',
    CHECKPOINT, '500', '256',
], cwd=REPO, env=RUNTIME_ENV)


## 10. Deterministic six-command video


In [ ]:
run([
    PYTHON,
    REPO / 'scripts' / 'play_rsl.py',
    '--task', 'DICE-Shadow-Play-v0',
    '--checkpoint', CHECKPOINT,
    '--output', REPO / 'videos' / 'DICE',
], cwd=REPO, env=RUNTIME_ENV)
